# 01 — Exploration

Sanity checks on the warehouse before doing any real analysis: row counts, null rates on the
columns the business questions depend on, and date-range coverage. Connects via SQLAlchemy to
the same BigQuery project the dashboard reads from — no separate data copy.

See `docs/architecture/README.md` for what each dataset holds:
- `olist_raw` — one table per source CSV, near-verbatim (dlt-managed, not queried directly below)
- `olist_warehouse` — staging views + star schema (`fact_orders`, `dim_customer`, `dim_date`)
- `olist_reporting` — the pre-aggregated datamart the dashboard reads

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

REPO_ROOT = Path.cwd().parent
load_dotenv(REPO_ROOT / "config" / "credentials.env")

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
engine = create_engine(f"bigquery://{PROJECT_ID}")

def q(sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

PROJECT_ID

'ntu-bigdata-project'

## Row counts

One row per table across all three datasets — first thing to check after any pipeline run.

In [6]:
tables = {
    "olist_warehouse": ["stg_orders", "stg_order_items", "stg_order_reviews", "stg_customers",
                         "int_order_delivery_stages", "dim_customer", "dim_date", "fact_orders"],
    "olist_reporting": ["mart_delivery_kpis", "mart_satisfaction_by_delivery", "mart_monthly_sales"],
}

counts = []
for dataset, table_list in tables.items():
    for table in table_list:
        n = q(f"SELECT COUNT(*) AS n FROM `{PROJECT_ID}.{dataset}.{table}`")["n"].iloc[0]
        counts.append({"dataset": dataset, "table": table, "row_count": n})

row_counts = pd.DataFrame(counts)
row_counts

,dataset,table,row_count
0,olist_warehouse,stg_orders,99441
1,olist_warehouse,stg_order_items,112650
2,olist_warehouse,stg_order_reviews,99224
3,olist_warehouse,stg_customers,99441
4,olist_warehouse,int_order_delivery_stages,99441
5,olist_warehouse,dim_customer,99441
6,olist_warehouse,dim_date,1096
7,olist_warehouse,fact_orders,99441
8,olist_reporting,mart_delivery_kpis,565
9,olist_reporting,mart_satisfaction_by_delivery,6


## Null checks on `fact_orders`

The columns the delay-math and satisfaction analysis depend on. Some nulls are expected here (e.g. undelivered orders have no `order_delivered_customer_at`) — this is about knowing the *rate*, not assuming zero.

In [7]:
null_check_sql = """
SELECT
    COUNT(*) AS total_rows,
    COUNTIF(order_purchase_at IS NULL) AS null_purchase_at,
    COUNTIF(order_approved_at IS NULL) AS null_approved_at,
    COUNTIF(order_delivered_carrier_at IS NULL) AS null_delivered_carrier_at,
    COUNTIF(order_delivered_customer_at IS NULL) AS null_delivered_customer_at,
    COUNTIF(order_estimated_delivery_at IS NULL) AS null_estimated_delivery_at,
    COUNTIF(avg_review_score IS NULL) AS null_avg_review_score,
    COUNTIF(delivery_status IS NULL) AS null_delivery_status
FROM `{project}.olist_warehouse.fact_orders`
""".format(project=PROJECT_ID)

null_counts = q(null_check_sql)
null_pct = null_counts.T.rename(columns={0: "count"})
null_pct["pct_of_total"] = (null_pct["count"] / null_counts["total_rows"].iloc[0] * 100).round(1)
null_pct

,count,pct_of_total
total_rows,99441,100.0
null_purchase_at,0,0.0
null_approved_at,160,0.2
null_delivered_carrier_at,1783,1.8
null_delivered_customer_at,2965,3.0
null_estimated_delivery_at,0,0.0
null_avg_review_score,768,0.8
null_delivery_status,0,0.0


## Date range coverage

Confirms the order history window and that timestamps landed as real `TIMESTAMP` types, not strings.

In [8]:
q(f"""
SELECT
    MIN(order_purchase_at) AS earliest_order,
    MAX(order_purchase_at) AS latest_order,
    MIN(order_delivered_customer_at) AS earliest_delivery,
    MAX(order_delivered_customer_at) AS latest_delivery
FROM `{PROJECT_ID}.olist_warehouse.fact_orders`
""")

,earliest_order,latest_order,earliest_delivery,latest_delivery
0,2016-09-04 21:15:19+00:00,2018-10-17 17:30:18+00:00,2016-10-11 13:46:32+00:00,2018-10-17 13:22:46+00:00


## Quick sanity: delivery status distribution & review score

Cross-check against the business case doc's hypothesis (late deliveries -> lower reviews) before doing the deeper analysis in `04_satisfaction_analysis.ipynb`.

In [9]:
status_order = ["early", "on_time", "late_1_3_days", "late_4_7_days", "late_8_plus_days", "not_delivered"]
satisfaction = q(f"SELECT * FROM `{PROJECT_ID}.olist_reporting.mart_satisfaction_by_delivery`")
satisfaction["delivery_status"] = pd.Categorical(satisfaction["delivery_status"], categories=status_order, ordered=True)
satisfaction.sort_values("delivery_status").reset_index(drop=True)

,delivery_status,order_count,avg_review_score,avg_delivery_days,avg_delivery_delay_days
0,early,86718,4.296428,10.756114,-13.204175
1,on_time,1450,4.157931,16.909885,-0.273563
2,late_1_3_days,3132,3.595307,21.610100,1.742603
3,late_4_7_days,1748,2.105549,28.188215,6.189049
4,late_8_plus_days,2782,1.698059,44.323793,20.106593
5,not_delivered,2843,1.753254,NaN,NaN
